# Instance-Aware Person Removal: Single-Shot vs Progressive Inpainting
## CSE-498R Directed Research — Full Pipeline (200 COCO Images)

---

### পুরো Pipeline:
```
Dataset (200 COCO images)
        ↓
YOLOv8x-seg  →  Person Detection + Segmentation Mask (একসাথে)
        ↓
Method A: Union Mask → LaMa Inpainting (একবার)
Method B: Per-Person → LaMa Inpainting (ধাপে ধাপে, largest-first)
        ↓
PSNR / SSIM / LPIPS Metrics
        ↓
CSV checkpoint + Visual comparisons + Aggregate charts
```

### মডেল পরিচিতি:
- **YOLOv8x-seg**: Detection + segmentation একসাথে (SAM ছাড়াই pixel-level mask)
- **LaMa (Large Mask inpainting)**: Samsung Research-এর state-of-the-art fast inpainter
  - SDXL-এর চেয়ে ~10x দ্রুত, T4-এ 200 ছবি ~3-4 ঘণ্টায় শেষ
  - Repetitive texture inpainting-এ SDXL-এর চেয়ে SSIM-এ ভালো

---
**Dataset path:** `/kaggle/input/coco-200-persons/`  
**Output path:** `/kaggle/working/results/`

## Cell 1 ─ Install Dependencies

In [ ]:
%%time
!pip install ultralytics simple-lama-inpainting lpips scikit-image opencv-python-headless -q

import os, sys
print(f'Python: {sys.version}')
print('Cell 1 complete ✅')

## Cell 2 ─ Imports & GPU Check

In [ ]:
import cv2
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')         # headless — no display server needed
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from pathlib import Path
import warnings, os, time, csv, traceback, json, random
from collections import Counter
warnings.filterwarnings('ignore')

from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity  as ssim_fn
import lpips

from ultralytics import YOLO
from simple_lama_inpainting import SimpleLama

# ── Device info ────────────────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU    : {props.name}')
    print(f'VRAM   : {props.total_memory/1e9:.1f} GB')
else:
    print('⚠️  No GPU — make sure Kaggle Accelerator is set to T4')

print('Cell 2 complete ✅')

## Cell 3 ─ Load Models

In [ ]:
%%time
# ── YOLOv8x-seg: Detection + Segmentation in one model ────────────────────
print('Loading YOLOv8x-seg...')
yolo_seg = YOLO('yolov8x-seg.pt')    # auto-download on first run
print(f'  YOLOv8x-seg loaded')

# ── LaMa: Fast, high-quality inpainting ────────────────────────────────────
print('Loading LaMa inpainting model...')
lama = SimpleLama()                   # auto-download weights ~200 MB
print(f'  LaMa loaded')

# ── LPIPS: Perceptual quality metric ──────────────────────────────────────
print('Loading LPIPS (AlexNet)...')
lpips_fn = lpips.LPIPS(net='alex').to(DEVICE)
print(f'  LPIPS loaded')

print('\n✅ All models ready!')

## Cell 4 ─ Configuration  ⚙️

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  ⚙️  CONFIGURATION
# ══════════════════════════════════════════════════════════════════

# Dataset path — Kaggle-এ Add Data করার পর এখানে সঠিক path দাও
IMAGE_DIR  = '/kaggle/input/coco-200-persons'   # ← dataset slug অনুযায়ী বদলাও

# Output
SAVE_DIR   = '/kaggle/working/results'
VISUAL_DIR = f'{SAVE_DIR}/visuals'
CSV_PATH   = f'{SAVE_DIR}/metrics.csv'

# Detection
YOLO_CONF       = 0.40    # confidence threshold
MAX_PERSONS     = 8       # per image
MIN_MASK_AREA   = 2000    # ignore tiny detections (pixels)

# Visuals — কতগুলো ছবির জন্য full comparison image save হবে
SAVE_VISUALS_N     = 30
RANDOM_VISUAL_SEED = 42

# Mask dilation (SAM-এর বদলে YOLO mask smooth করতে)
MASK_DILATE_KERNEL = 15   # pixels
MASK_BLUR_RADIUS   = 21   # feathering

# Image extensions
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
# ══════════════════════════════════════════════════════════════════

os.makedirs(SAVE_DIR,   exist_ok=True)
os.makedirs(VISUAL_DIR, exist_ok=True)

# Image list
all_images = sorted([
    p for p in Path(IMAGE_DIR).rglob('*')
    if p.suffix.lower() in IMG_EXTS
])
print(f'Images found : {len(all_images)}')
print(f'First 3      : {[p.name for p in all_images[:3]]}')

# Visual subset selection
rng = random.Random(RANDOM_VISUAL_SEED)
visual_set = set(map(str, rng.sample(all_images, min(SAVE_VISUALS_N, len(all_images)))))
print(f'Will save visual comparisons for : {len(visual_set)} images')

# CSV — resume support
CSV_FIELDS = [
    'image','n_persons',
    'A_PSNR','A_SSIM','A_LPIPS','A_mask_pct','A_time_s',
    'B_PSNR','B_SSIM','B_LPIPS','B_mask_pct','B_time_s',
    'status'
]

already_done = set()
if os.path.exists(CSV_PATH):
    with open(CSV_PATH) as f:
        for row in csv.DictReader(f):
            if row.get('status') == 'ok':
                already_done.add(row['image'])
    print(f'Resume: {len(already_done)} already done, '
          f'{len(all_images)-len(already_done)} remaining')
else:
    with open(CSV_PATH, 'w', newline='') as f:
        csv.DictWriter(f, fieldnames=CSV_FIELDS).writeheader()
    print('New CSV created')

print('\nCell 4 complete ✅')

## Cell 5 ─ Core Functions

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
#  DETECTION + SEGMENTATION (YOLOv8-seg)
# ──────────────────────────────────────────────────────────────────────────
def detect_and_segment(image_bgr):
    """
    YOLOv8x-seg দিয়ে একসাথে detect + segment করে।
    Returns: list of dicts with 'bbox', 'mask', 'area'
    """
    H, W = image_bgr.shape[:2]
    results = yolo_seg(image_bgr, conf=YOLO_CONF, classes=[0], verbose=False)[0]

    persons = []
    if results.masks is None or results.boxes is None:
        return persons

    masks_data = results.masks.data.cpu().numpy()   # (N, H_r, W_r)
    boxes_data = results.boxes.xyxy.cpu().numpy()   # (N, 4)

    for i in range(len(boxes_data)):
        # Resize mask to original image size
        mask_small = masks_data[i]                  # float 0-1
        mask_full  = cv2.resize(
            mask_small, (W, H), interpolation=cv2.INTER_LINEAR
        )
        binary = (mask_full > 0.5).astype(np.uint8) * 255

        area = int(binary.sum() / 255)
        if area < MIN_MASK_AREA:
            continue   # too small — skip

        x1, y1, x2, y2 = map(int, boxes_data[i])
        persons.append({
            'bbox': [x1, y1, x2, y2],
            'mask': binary,
            'area': area,
        })

    # Largest first
    persons.sort(key=lambda p: p['area'], reverse=True)
    return persons[:MAX_PERSONS]


# ──────────────────────────────────────────────────────────────────────────
#  MASK POST-PROCESSING
# ──────────────────────────────────────────────────────────────────────────
def dilate_and_feather(mask):
    """Dilate slightly + feather edges for natural inpainting boundary."""
    kernel  = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (MASK_DILATE_KERNEL, MASK_DILATE_KERNEL)
    )
    dilated  = cv2.dilate(mask, kernel, iterations=1)
    feathered = cv2.GaussianBlur(
        dilated.astype(np.float32),
        (MASK_BLUR_RADIUS, MASK_BLUR_RADIUS), 0
    )
    return (feathered > 127).astype(np.uint8) * 255


# ──────────────────────────────────────────────────────────────────────────
#  LAMA INPAINTING
# ──────────────────────────────────────────────────────────────────────────
def lama_inpaint(image_rgb, mask):
    """
    LaMa দিয়ে inpainting করে।
    image_rgb: np.uint8 (H,W,3)
    mask     : np.uint8 (H,W) — 255=inpaint here, 0=keep
    Returns  : np.uint8 (H,W,3)
    """
    pil_img  = Image.fromarray(image_rgb)
    pil_mask = Image.fromarray(mask)

    result = lama(pil_img, pil_mask)   # returns PIL Image

    out_np = np.array(result.convert('RGB'))
    # Composite: only replace masked area
    mask_3 = np.stack([mask / 255.0] * 3, axis=-1)
    return (out_np * mask_3 + image_rgb * (1 - mask_3)).astype(np.uint8)


# ──────────────────────────────────────────────────────────────────────────
#  METHOD A — SINGLE-SHOT
# ──────────────────────────────────────────────────────────────────────────
def method_single_shot(image_rgb, persons):
    """
    সব person-এর mask একসাথে merge করে → একটা LaMa call।
    """
    H, W = image_rgb.shape[:2]
    union_mask = np.zeros((H, W), dtype=np.uint8)
    for p in persons:
        union_mask = np.maximum(union_mask, dilate_and_feather(p['mask']))

    t0     = time.time()
    result = lama_inpaint(image_rgb, union_mask)
    return {
        'result':     result,
        'union_mask': union_mask,
        'time_s':     round(time.time() - t0, 2)
    }


# ──────────────────────────────────────────────────────────────────────────
#  METHOD B — PROGRESSIVE (largest-first)
# ──────────────────────────────────────────────────────────────────────────
def method_progressive(image_rgb, persons):
    """
    একটা একটা করে person সরায় — প্রতিটা result পরের step-এর input হয়।
    Largest person সবার আগে সরানো হয়।
    """
    H, W    = image_rgb.shape[:2]
    current  = image_rgb.copy()
    cum_mask = np.zeros((H, W), dtype=np.uint8)
    steps    = []
    t0       = time.time()

    for i, p in enumerate(persons):
        step_mask = dilate_and_feather(p['mask'])
        cum_mask  = np.maximum(cum_mask, step_mask)
        current   = lama_inpaint(current, step_mask)
        steps.append(current.copy())

    return {
        'result':      current,
        'union_mask':  cum_mask,
        'step_images': steps,
        'time_s':      round(time.time() - t0, 2)
    }


# ──────────────────────────────────────────────────────────────────────────
#  METRICS  (masked region only)
# ──────────────────────────────────────────────────────────────────────────
def compute_metrics(original, reconstructed, mask):
    m = mask > 127
    if not m.any():
        return {'PSNR': 0.0, 'SSIM': 0.0, 'LPIPS': 1.0, 'mask_coverage_%': 0.0}

    ys, xs  = np.where(m)
    y1, y2  = ys.min(), ys.max()
    x1, x2  = xs.min(), xs.max()
    orig_c  = original[y1:y2, x1:x2]
    recon_c = reconstructed[y1:y2, x1:x2]

    if orig_c.shape[0] < 8 or orig_c.shape[1] < 8:
        return {'PSNR': 0.0, 'SSIM': 0.0, 'LPIPS': 1.0, 'mask_coverage_%': 0.0}

    psnr_val = psnr_fn(orig_c, recon_c, data_range=255)
    ssim_val = ssim_fn(orig_c, recon_c, channel_axis=2, data_range=255)

    def to_tensor(img):
        t = torch.from_numpy(img.astype(np.float32) / 127.5 - 1.0)
        return t.permute(2, 0, 1).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        lp = lpips_fn(to_tensor(orig_c), to_tensor(recon_c)).item()

    return {
        'PSNR':            round(psnr_val, 3),
        'SSIM':            round(ssim_val, 4),
        'LPIPS':           round(lp, 4),
        'mask_coverage_%': round(100.0 * m.sum() / m.size, 2),
    }


# ──────────────────────────────────────────────────────────────────────────
#  VISUAL SAVE
# ──────────────────────────────────────────────────────────────────────────
def save_visual(stem, original, persons, res_A, res_B, mA, mB):
    # ── 4-panel main comparison ────────────────────────────────────────────
    fig, axes = plt.subplots(1, 4, figsize=(24, 6))
    fig.suptitle(
        f'{stem}  |  {len(persons)} person(s) detected',
        fontsize=12, fontweight='bold'
    )

    # Overlay boxes on original
    vis = original.copy()
    for i, p in enumerate(persons):
        x1, y1, x2, y2 = p['bbox']
        cv2.rectangle(vis, (x1,y1), (x2,y2), (0,220,80), 3)
        cv2.putText(vis, f'P{i+1}', (x1+4, y1+36),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,220,80), 2)

    # Segmentation mask overlay
    seg_vis = original.copy().astype(np.float32)
    for p in persons:
        color_layer           = np.zeros_like(original, dtype=np.float32)
        color_layer[:,:,1]    = p['mask'].astype(np.float32)  # green
        alpha_mask            = (p['mask'] / 255.0)[:, :, None]
        seg_vis = seg_vis * (1 - 0.45*alpha_mask) + color_layer * 0.45*alpha_mask
    seg_vis = seg_vis.clip(0, 255).astype(np.uint8)

    def show(ax, img, title, sub=''):
        ax.imshow(img)
        ax.set_title(f'{title}\n{sub}', fontsize=8)
        ax.axis('off')

    show(axes[0], original, 'Original')
    show(axes[1], seg_vis,  'YOLOv8-seg Masks', f'{len(persons)} person(s)')
    show(axes[2], res_A['result'], 'Method A — Single-Shot',
         f"PSNR={mA['PSNR']} | SSIM={mA['SSIM']} | LPIPS={mA['LPIPS']}\n"
         f"mask={mA['mask_coverage_%']}% | {res_A['time_s']}s")
    show(axes[3], res_B['result'], 'Method B — Progressive',
         f"PSNR={mB['PSNR']} | SSIM={mB['SSIM']} | LPIPS={mB['LPIPS']}\n"
         f"mask={mB['mask_coverage_%']}% | {res_B['time_s']}s")

    plt.tight_layout()
    plt.savefig(f'{VISUAL_DIR}/{stem}_comparison.jpg', dpi=110, bbox_inches='tight')
    plt.close(fig)

    # ── Progressive strip (only if ≥ 2 persons) ────────────────────────────
    steps = res_B.get('step_images', [])
    if len(steps) >= 2:
        n_cols = len(steps) + 1
        fig2, axes2 = plt.subplots(1, n_cols, figsize=(6*n_cols, 5))
        fig2.suptitle('Method B — Progressive Steps (largest person first)',
                      fontsize=11, fontweight='bold')
        axes2[0].imshow(original); axes2[0].set_title('Original'); axes2[0].axis('off')
        for j, s in enumerate(steps):
            axes2[j+1].imshow(s)
            axes2[j+1].set_title(f'After P{j+1} removed')
            axes2[j+1].axis('off')
        plt.tight_layout()
        plt.savefig(f'{VISUAL_DIR}/{stem}_progressive.jpg', dpi=100, bbox_inches='tight')
        plt.close(fig2)


# ──────────────────────────────────────────────────────────────────────────
#  CSV APPEND (crash-safe)
# ──────────────────────────────────────────────────────────────────────────
def append_csv(row_dict):
    with open(CSV_PATH, 'a', newline='') as f:
        csv.DictWriter(f, fieldnames=CSV_FIELDS).writerow(row_dict)


print('Cell 5 complete ✅  — all functions defined')

## Cell 6 ─ 🚀 Main Batch Loop
এই cell চালালেই সব 200 ছবি automatically process হবে।  
Session timeout হলে আবার চালাও — already done ছবিগুলো skip হবে।

In [ ]:
total              = len(all_images)
skipped_no_person  = 0
errors             = []
t_global_start     = time.time()

print(f'Total images       : {total}')
print(f'Already processed  : {len(already_done)}')
print(f'To process now     : {total - len(already_done)}')
print('='*72)

for idx, img_path in enumerate(all_images, 1):
    name = img_path.name
    stem = img_path.stem

    # ── Resume check ──────────────────────────────────────────────────────
    if name in already_done:
        print(f'[{idx:>3}/{total}] SKIP (done): {name}')
        continue

    print(f'\n[{idx:>3}/{total}] {name}')
    t_img = time.time()

    try:
        # ── Load ────────────────────────────────────────────────────────
        image_bgr = cv2.imread(str(img_path))
        if image_bgr is None:
            raise ValueError('cv2.imread returned None')
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        H, W = image_rgb.shape[:2]
        print(f'  Size: {W}×{H}')

        # ── Detection + Segmentation ────────────────────────────────────
        persons = detect_and_segment(image_bgr)
        n = len(persons)
        print(f'  Persons detected: {n}')

        if n == 0:
            append_csv({'image': name, 'n_persons': 0,
                'A_PSNR':'-','A_SSIM':'-','A_LPIPS':'-',
                'A_mask_pct':'-','A_time_s':'-',
                'B_PSNR':'-','B_SSIM':'-','B_LPIPS':'-',
                'B_mask_pct':'-','B_time_s':'-',
                'status':'no_person'})
            skipped_no_person += 1
            print('  ⚠ No person detected — logged and skipped')
            continue

        # ── Method A: Single-Shot ────────────────────────────────────────
        print('  [A] Single-Shot...', end=' ', flush=True)
        res_A = method_single_shot(image_rgb, persons)
        mA    = compute_metrics(image_rgb, res_A['result'], res_A['union_mask'])
        print(f"PSNR={mA['PSNR']} SSIM={mA['SSIM']} LPIPS={mA['LPIPS']} | {res_A['time_s']}s")

        # ── Method B: Progressive ────────────────────────────────────────
        print(f'  [B] Progressive ({n} steps)...', end=' ', flush=True)
        res_B = method_progressive(image_rgb, persons)
        mB    = compute_metrics(image_rgb, res_B['result'], res_B['union_mask'])
        print(f"PSNR={mB['PSNR']} SSIM={mB['SSIM']} LPIPS={mB['LPIPS']} | {res_B['time_s']}s")

        # ── Visual (subset only) ─────────────────────────────────────────
        if str(img_path) in visual_set:
            save_visual(stem, image_rgb, persons, res_A, res_B, mA, mB)
            print('  Visual saved')

        # ── CSV checkpoint ────────────────────────────────────────────────
        append_csv({
            'image': name, 'n_persons': n,
            'A_PSNR':  mA['PSNR'],  'A_SSIM':  mA['SSIM'],
            'A_LPIPS': mA['LPIPS'], 'A_mask_pct': mA['mask_coverage_%'],
            'A_time_s': res_A['time_s'],
            'B_PSNR':  mB['PSNR'],  'B_SSIM':  mB['SSIM'],
            'B_LPIPS': mB['LPIPS'], 'B_mask_pct': mB['mask_coverage_%'],
            'B_time_s': res_B['time_s'],
            'status': 'ok'
        })

        elapsed  = time.time() - t_img
        done_now = idx - len(already_done) - skipped_no_person - len(errors)
        rem      = total - idx
        eta_min  = (time.time()-t_global_start) / max(done_now,1) * rem / 60
        print(f'  ✅ {elapsed:.1f}s | ETA ≈ {eta_min:.0f} min remaining')

        del res_A, res_B
        torch.cuda.empty_cache()

    except Exception as e:
        print(f'  ❌ ERROR: {e}')
        traceback.print_exc()
        errors.append({'image': name, 'error': str(e)})
        append_csv({'image': name, 'n_persons': -1,
            'A_PSNR':'-','A_SSIM':'-','A_LPIPS':'-',
            'A_mask_pct':'-','A_time_s':'-',
            'B_PSNR':'-','B_SSIM':'-','B_LPIPS':'-',
            'B_mask_pct':'-','B_time_s':'-',
            'status': 'error'})
        torch.cuda.empty_cache()

total_min = (time.time() - t_global_start) / 60
print('\n' + '='*72)
print(f'BATCH COMPLETE  ({total_min:.1f} min total)')
print(f'  Processed successfully : {total - len(already_done) - skipped_no_person - len(errors)}')
print(f'  No person (skipped)    : {skipped_no_person}')
print(f'  Errors                 : {len(errors)}')
print(f'  Results CSV            : {CSV_PATH}')
print('='*72)

## Cell 7 ─ 📊 Full Analysis & Charts

In [ ]:
# ── Load valid results ─────────────────────────────────────────────────────
rows = []
with open(CSV_PATH) as f:
    for row in csv.DictReader(f):
        if row['status'] == 'ok':
            rows.append(row)

print(f'Valid results: {len(rows)} images')

def col(key): return np.array([float(r[key]) for r in rows])

A_psnr  = col('A_PSNR');  B_psnr  = col('B_PSNR')
A_ssim  = col('A_SSIM');  B_ssim  = col('B_SSIM')
A_lpips = col('A_LPIPS'); B_lpips = col('B_LPIPS')
A_time  = col('A_time_s');B_time  = col('B_time_s')
n_pers  = col('n_persons')

# ── CONSOLE SUMMARY TABLE ──────────────────────────────────────────────────
print('\n' + '='*74)
print('  AGGREGATE RESULTS  —  200 COCO Images')
print('='*74)
hdr = f'{"Metric":<22} {"Method A (Single-Shot)":>22} {"Method B (Progressive)":>22}  Winner'
print(hdr); print('-'*74)

summary_data = [
    ('PSNR  (↑ better)', A_psnr,  B_psnr,  'high'),
    ('SSIM  (↑ better)', A_ssim,  B_ssim,  'high'),
    ('LPIPS (↓ better)', A_lpips, B_lpips, 'low'),
]
for label, a, b, direction in summary_data:
    am, bm = a.mean(), b.mean()
    as_, bs_ = a.std(), b.std()
    if direction == 'high':
        winner = '✅ B-Progressive' if bm > am else '✅ A-SingleShot'
    else:
        winner = '✅ B-Progressive' if bm < am else '✅ A-SingleShot'
    print(f'{label:<22} {am:>10.4f} ± {as_:<8.4f} {bm:>10.4f} ± {bs_:<8.4f}  {winner}')

print('-'*74)
print(f'{"Avg time/image (s)":<22} {A_time.mean():>22.2f} {B_time.mean():>22.2f}')
print(f'{"Total time (min)":<22} {A_time.sum()/60:>22.1f} {B_time.sum()/60:>22.1f}')
print('='*74)
print(f'N images  : {len(rows)}')
print(f'Avg persons/image: {n_pers.mean():.2f}  |  '
      f'Max: {int(n_pers.max())}  |  Min: {int(n_pers.min())}')
print('='*74)

In [ ]:
# ── FIGURE 1: Box Plot (distribution) ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.suptitle(
    f'Method A (Single-Shot) vs Method B (Progressive)\n'
    f'N = {len(rows)} images from COCO 2017 val set',
    fontsize=13, fontweight='bold'
)
CA, CB = '#e07070', '#5b9bd5'

for ax, (a_arr, b_arr, title) in zip(axes, [
    (A_psnr,  B_psnr,  'PSNR (dB)  ↑ higher is better'),
    (A_ssim,  B_ssim,  'SSIM  ↑ higher is better'),
    (A_lpips, B_lpips, 'LPIPS  ↓ lower is better'),
]):
    bp = ax.boxplot([a_arr, b_arr],
                    labels=['A: Single-Shot', 'B: Progressive'],
                    patch_artist=True, notch=True,
                    medianprops=dict(color='black', linewidth=2.5))
    for patch, c in zip(bp['boxes'], [CA, CB]):
        patch.set_facecolor(c); patch.set_alpha(0.75)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    for i, arr in enumerate([a_arr, b_arr], 1):
        ax.text(i, arr.max() * 1.01, f'μ={arr.mean():.3f}',
                ha='center', fontsize=10, fontweight='bold',
                color=[CA, CB][i-1])

plt.tight_layout()
p = f'{SAVE_DIR}/fig1_boxplot.png'
plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved: {p}')

In [ ]:
# ── FIGURE 2: Win-Rate Pie Charts ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle('Per-Image Win Rate: Method A vs Method B', fontsize=13, fontweight='bold')

for ax, (a_arr, b_arr, metric, direction) in zip(axes, [
    (A_psnr,  B_psnr,  'PSNR',  'high'),
    (A_ssim,  B_ssim,  'SSIM',  'high'),
    (A_lpips, B_lpips, 'LPIPS', 'low'),
]):
    wins_B = int((b_arr > a_arr).sum()) if direction=='high' else int((b_arr < a_arr).sum())
    wins_A = len(rows) - wins_B
    wedge_props = dict(edgecolor='white', linewidth=2)
    ax.pie(
        [wins_A, wins_B],
        labels=[f'A wins\n({wins_A}/{len(rows)})', f'B wins\n({wins_B}/{len(rows)})'],
        colors=[CA, CB], autopct='%1.1f%%', startangle=90,
        textprops={'fontsize': 12}, wedgeprops=wedge_props
    )
    ax.set_title(metric, fontsize=13, fontweight='bold')

plt.tight_layout()
p = f'{SAVE_DIR}/fig2_win_rate.png'
plt.savefig(p, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved: {p}')

In [ ]:
# ── FIGURE 3: Metric vs Number of Persons ─────────────────────────────────
unique_n = sorted(set(n_pers.astype(int)))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Metric vs Number of Persons per Image', fontsize=13, fontweight='bold')

for ax, (a_arr, b_arr, ylabel, direction) in zip(axes, [
    (A_psnr,  B_psnr,  'PSNR (dB)', 'high'),
    (A_ssim,  B_ssim,  'SSIM',      'high'),
    (A_lpips, B_lpips, 'LPIPS',     'low'),
]):
    for n_val in unique_n:
        mask = n_pers.astype(int) == n_val
        if not mask.any(): continue
        ax.scatter([n_val - 0.12]*mask.sum(), a_arr[mask],
                   c=CA, alpha=0.55, s=30, label='A' if n_val==unique_n[0] else '')
        ax.scatter([n_val + 0.12]*mask.sum(), b_arr[mask],
                   c=CB, alpha=0.55, s=30, label='B' if n_val==unique_n[0] else '')

    # Mean lines
    grp_a = [a_arr[n_pers.astype(int)==n].mean() for n in unique_n
             if (n_pers.astype(int)==n).any()]
    grp_b = [b_arr[n_pers.astype(int)==n].mean() for n in unique_n
             if (n_pers.astype(int)==n).any()]
    valid_n = [n for n in unique_n if (n_pers.astype(int)==n).any()]

    ax.plot(valid_n, grp_a, 'o--', color='#a00000', lw=2.0, label='A mean')
    ax.plot(valid_n, grp_b, 's--', color='#1a4f8a', lw=2.0, label='B mean')

    ax.set_xlabel('# persons in image', fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_xticks(unique_n)
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
p = f'{SAVE_DIR}/fig3_metric_vs_persons.png'
plt.savefig(p, dpi=140, bbox_inches='tight'); plt.show()
print(f'Saved: {p}')

In [ ]:
# ── FIGURE 4: Per-image bar (sorted by Method A PSNR) ─────────────────────
sort_idx = np.argsort(A_psnr)
x = np.arange(len(rows)); w = 0.4

fig, axes = plt.subplots(3, 1, figsize=(max(22, len(rows)*0.28), 15), sharex=True)
fig.suptitle('Per-Image Metrics — Sorted by Method A PSNR', fontsize=12, fontweight='bold')

for ax, (a_arr, b_arr, ylabel) in zip(axes, [
    (A_psnr[sort_idx],  B_psnr[sort_idx],  'PSNR (dB)'),
    (A_ssim[sort_idx],  B_ssim[sort_idx],  'SSIM'),
    (A_lpips[sort_idx], B_lpips[sort_idx], 'LPIPS'),
]):
    ax.bar(x-w/2, a_arr, w, color=CA, alpha=0.80, label='A: Single-Shot')
    ax.bar(x+w/2, b_arr, w, color=CB, alpha=0.80, label='B: Progressive')
    ax.axhline(a_arr.mean(), color='#900000', lw=1.8, ls='--', alpha=0.8,
               label=f'A mean={a_arr.mean():.3f}')
    ax.axhline(b_arr.mean(), color='#14406b', lw=1.8, ls='--', alpha=0.8,
               label=f'B mean={b_arr.mean():.3f}')
    ax.set_ylabel(ylabel, fontsize=10)
    ax.legend(fontsize=8, loc='upper left'); ax.grid(axis='y', alpha=0.25)

axes[-1].set_xlabel('Image index (sorted by A:PSNR)', fontsize=10)
plt.tight_layout()
p = f'{SAVE_DIR}/fig4_per_image_bars.png'
plt.savefig(p, dpi=130, bbox_inches='tight'); plt.show()
print(f'Saved: {p}')

In [ ]:
# ── FIGURE 5: Scatter (PSNR vs SSIM, PSNR vs LPIPS) ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Correlation Scatter Plots', fontsize=13, fontweight='bold')

axes[0].scatter(A_psnr, A_ssim, c=CA, alpha=0.6, s=35, label='A: Single-Shot')
axes[0].scatter(B_psnr, B_ssim, c=CB, alpha=0.6, s=35, label='B: Progressive')
axes[0].set_xlabel('PSNR (dB)'); axes[0].set_ylabel('SSIM')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_title('PSNR vs SSIM')

axes[1].scatter(A_psnr, A_lpips, c=CA, alpha=0.6, s=35, label='A: Single-Shot')
axes[1].scatter(B_psnr, B_lpips, c=CB, alpha=0.6, s=35, label='B: Progressive')
axes[1].set_xlabel('PSNR (dB)'); axes[1].set_ylabel('LPIPS')
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_title('PSNR vs LPIPS')

plt.tight_layout()
p = f'{SAVE_DIR}/fig5_scatter.png'
plt.savefig(p, dpi=140, bbox_inches='tight'); plt.show()
print(f'Saved: {p}')

print(f'\n✅ All 5 figures saved to: {SAVE_DIR}')
print(f'Visual comparisons in    : {VISUAL_DIR}')

## Cell 8 ─ 🖼️ Visual Grid (sample results)

In [ ]:
# Show a 3×3 grid of comparison images saved during batch
saved = list(Path(VISUAL_DIR).glob('*_comparison.jpg'))
print(f'Saved comparison images: {len(saved)}')

if len(saved) >= 1:
    SHOW = min(9, len(saved))
    sample = random.sample(saved, SHOW)
    cols   = 3
    rows_n = (SHOW + cols - 1) // cols

    fig, axes = plt.subplots(rows_n, cols, figsize=(cols*8, rows_n*4))
    fig.suptitle('Sample Visual Results (Original | Seg Mask | Method A | Method B)',
                 fontsize=12, fontweight='bold')
    axes_flat = axes.flatten() if SHOW > 1 else [axes]

    for ax, fp in zip(axes_flat, sample):
        img = plt.imread(str(fp))
        ax.imshow(img)
        ax.set_title(fp.stem[:50], fontsize=6)
        ax.axis('off')
    for ax in axes_flat[len(sample):]:
        ax.axis('off')

    plt.tight_layout()
    pg = f'{SAVE_DIR}/sample_grid.png'
    plt.savefig(pg, dpi=100, bbox_inches='tight')
    plt.show()
    print(f'Saved: {pg}')